# 学習したTTSモデルを使って音声を合成する
このファイルを`espnet/egs2/○○/tts1/`フォルダに置いて、上のブロックから順番に実行する

In [40]:
from espnet2.bin.tts_inference import Text2Speech
from espnet2.utils.types import str_or_none

モデルと設定ファイルのパスを記述する<br>
妙な処理をしていなければどちらのファイルも`exp/tts_train_○○/`にあるはず

In [41]:
# /home/{ユーザー名}/espnet2/egs2/{レシピ名}/tts1
%cd /home/uesugi/espnet-for-customize-jets/egs2/jsut_d0_jets/tts1

import os

# 音声をwavファイルに出力するかどうか
save_wavfile = True
# 音声ファイルの保存にscipyかpysoundfile(sf)どちらを使うか
# save_method = "scipy"
save_method = "sf"
# モデルのパス
model = "/home/uesugi/espnet-for-customize-jets/egs2/jsut_d0_jets/tts1/exp/22.05k/tts_train_jets_22.05k_raw_phn_jaconv_pyopenjtalk_prosody/save/1000epoch.pth"
# 学習時の設定ファイルのパス (指定しなければモデルのパスから勝手に探す)
config = "/home/uesugi/espnet-for-customize-jets/egs2/jsut_d0_jets/tts1/exp/22.05k/tts_train_jets_22.05k_raw_phn_jaconv_pyopenjtalk_prosody/config.yaml"
if config == None or config == '':
    config = os.path.join(os.path.dirname(model), "config.yaml")


# colabのサンプルに書いてあったもの
# @title Choose Japanese model { run: "auto" }
# lang = 'Japanese'
# tag = 'kan-bayashi/jsut_full_band_vits_prosody' #@param ["kan-bayashi/jsut_tacotron2", "kan-bayashi/jsut_transformer", "kan-bayashi/jsut_fastspeech", "kan-bayashi/jsut_fastspeech2", "kan-bayashi/jsut_conformer_fastspeech2", "kan-bayashi/jsut_conformer_fastspeech2_accent", "kan-bayashi/jsut_conformer_fastspeech2_accent_with_pause", "kan-bayashi/jsut_vits_accent_with_pause", "kan-bayashi/jsut_full_band_vits_accent_with_pause", "kan-bayashi/jsut_tacotron2_prosody", "kan-bayashi/jsut_transformer_prosody", "kan-bayashi/jsut_conformer_fastspeech2_tacotron2_prosody", "kan-bayashi/jsut_vits_prosody", "kan-bayashi/jsut_full_band_vits_prosody", "kan-bayashi/jvs_jvs010_vits_prosody", "kan-bayashi/tsukuyomi_full_band_vits_prosody"] {type:"string"}
# vocoder_tag = 'none' #@param ["none", "parallel_wavegan/jsut_parallel_wavegan.v1", "parallel_wavegan/jsut_multi_band_melgan.v2", "parallel_wavegan/jsut_style_melgan.v1", "parallel_wavegan/jsut_hifigan.v1"] {type:"string"}

/home/uesugi/espnet-for-customize-jets/egs2/jsut_d0_jets/tts1


In [42]:
text2speech = Text2Speech.from_pretrained(
    model_file=model,
    train_config=config,
    # web上のモデルをダウンロードするときの設定
    # model_tag=str_or_none(tag),
    # vocoder_tag=str_or_none(vocoder_tag),
    device="cpu", # "cuda"にしてもいいけどcpuで十分な速度  
    # Only for Tacotron 2 & Transformer
    threshold=0.5,
    # Only for Tacotron 2
    minlenratio=0.0,
    maxlenratio=10.0,
    use_att_constraint=False,
    backward_window=1,
    forward_window=3,
    # Only for FastSpeech & FastSpeech2 & VITS
    speed_control_alpha=1.0,
    # Only for VITS
    noise_scale=0.333,
    noise_scale_dur=0.333,
    d0_control_alpha=1.0,
)

Debug - Filtered kwargs: {'model_file': '/home/uesugi/espnet-for-customize-jets/egs2/jsut_d0_jets/tts1/exp/22.05k/tts_train_jets_22.05k_raw_phn_jaconv_pyopenjtalk_prosody/save/1000epoch.pth', 'train_config': '/home/uesugi/espnet-for-customize-jets/egs2/jsut_d0_jets/tts1/exp/22.05k/tts_train_jets_22.05k_raw_phn_jaconv_pyopenjtalk_prosody/config.yaml', 'device': 'cpu', 'threshold': 0.5, 'minlenratio': 0.0, 'maxlenratio': 10.0, 'use_att_constraint': False, 'backward_window': 1, 'forward_window': 3, 'speed_control_alpha': 1.0, 'noise_scale': 0.333, 'noise_scale_dur': 0.333, 'd0_control_alpha': 1.0}
Debug - Argument types:
  use_teacher_forcing: <class 'bool'> = False
  d0_control_alpha: <class 'float'> = 1.0
  speed_control_alpha: <class 'float'> = 1.0
  noise_scale: <class 'float'> = 0.333
  always_fix_seed: <class 'bool'> = False


In [ ]:
# ─── 必要ライブラリ ─────────────────────────────────────────
from IPython.display import Audio, display
import torch, os, re
from pathlib import Path

# 保存方法の選択（既存フラグをそのまま利用）
if save_method == "sf":
    import soundfile as sf
elif save_method == "scipy":
    from scipy.io.wavfile import write

# ─── ① ここに好きな文を並べる ───────────────────────────────
texts = [
    "水をマレーシアから買わなくてはならないのです。",
    "木曜日、停戦会談は、何の進展もないまま終了しました。",
    "上院議員は私がデータをゆがめたと告発した。",
    "１週間して、そのニュースは本当になった。",
    "血圧は、健康のパロメーターとして重要である。",
    "週に四回、フランスの授業があります。",
    "許可書がなければここへは入れない。",
    "大声で泣きながら、女の子は母親を探していた。",
    "無罪の人々は、もちろん放免された。",
    "末期試験に備えて、本当に気合いを入れて勉強しなきゃ。",
    "木曽川は、しばしば日本のライン川と呼ばれている。",
    "溺れかかっていた乗客は、すべて救助された。",
    "中心部にあるので、商店や、オフィスに行くのに便利です。",
    "残酷ということは、彼の性質にはないことだ。",
    "庭園の周りに、ぐるりと高いへいが立っている。",
    "出生率と死亡率は、ほぼ等しかった。",
    "絶対にそのスイッチに触ってはいけない。",
    "部屋を出るときには、明かりを消しなさい。",
    "シェークスピアの作品は、とてもむずかしくて読めない。",
    "むかし、むかし、１人の老人が住んでいた。",
    "疲労やら、飢えやらで、彼は目眩を感じた。",
    "むしろロン毛のほうが禿げやすいって聞いたぞ。",
    "むこうに立っている女の子は、メアリーです。",
    "システィナ礼拝堂は、１４７３年に、バティカン宮殿内に建立された、壮大な礼拝堂です。",
    "テニスにもあるけど、４大大会って何。",
    "策略は衣服を必要とするが、真実は裸であることを好む。",
    "濃いコーヒーより、うすいコーヒーの方が好きです。",
    "外交官には、様々な特権が与えられている。",
    "刺し傷はとても深く、感染の恐れがないか検査する必要がある。",
    "外人が日本食に慣れることはむずかしい。",
    "語の古い意味が、現在の基本的な意味であるとは限らない。",
    "針金は、電気を伝えるのにもちいられる。",
    "来月の歌舞伎座の出し物はなんですか。",
    "圧縮したファイルを添付で送ってください。",
    "容姿端麗、頭脳明晰、運動神経抜群、家は金持ちで、ついでに学生会の副会長をしてたりもする、いわゆる、パーフェクトな奴だ。",
    "入院患者は、医者に麻酔を注射されて、すぐに眠りに落ちた。",
    "傍目八目という言葉があるように一度協会から離れて、日本サッカーをみて頂きたい。",
    "行儀の悪さは、彼の良識を疑わせるものだ。",
    "部長の都合が悪くなってしまったので、飲み会の日程は仕切り直しだね。",
    "蛇をみたとき、彼は悲鳴をあげた。",
    "紫外線は、皮膚癌を引き起こすことがある。",
    "気がつくと、逃げ場はどこにもなかった。",
    "三番目の、そしてもっとも重要な考えは、再入ということである。",
    "希望や夢の思いは、絶対に見つからない。",
    "前方にある、あのサインが読めますか。",
    "気まずい沈黙の後、ビルは彼女の手を取って、上の階へ引っ張って行った。",
    "鉱山労働者が、賃上げを要求してストに突入した。",
    "気の弱い男が、美女を得たためしがない。",
    "気前の良いその歯科医は、およそ２０億円を、慈善事業に寄付した。",
    "気に入ろうが入るまいが、君は行かねばならない。",  
]

# ─── ② 出力フォルダの用意 ───────────────────────────────
out_dir = Path("inference_for_eval_d0_control_1.0")
out_dir.mkdir(exist_ok=True)

# ─── ③ ループで推論・保存・再生 ──────────────────────────
for idx, text in enumerate(texts, start=1):
    # 推論
    with torch.no_grad():
        wav_tensor = text2speech(text)["wav"]
    audio_array = wav_tensor.view(-1).cpu().numpy()
    samplerate   = text2speech.fs

    # ファイル名生成（安全文字だけ残す）
    safe = re.sub(r"[^\w]", "_", text[:10])
    filename  = f"{idx:02d}_{safe}.wav"
    filepath  = out_dir / filename

    # 保存
    if save_wavfile:
        if save_method == "sf":
            sf.write(filepath, audio_array, samplerate)
        elif save_method == "scipy":
            write(filepath, samplerate, audio_array)

    # ノートブック上で再生
    print(f"▶️  {filename}")
    display(Audio(audio_array, rate=samplerate))

print(f"{len(texts)} 件の音声を生成しました。")


▶️  01_水をマレーシアから買.wav


▶️  02_木曜日_停戦会談は_.wav


▶️  03_上院議員は私がデータ.wav


▶️  04_１週間して_そのニュ.wav


▶️  05_血圧は_健康のパロメ.wav


...

▶️  07_許可書がなければここ.wav


▶️  08_大声で泣きながら_女.wav


▶️  09_無罪の人々は_もちろ.wav


▶️  10_末期試験に備えて_本.wav


▶️  11_木曽川は_しばしば日.wav
